# 05 Did-not-attend rebuild
I rebuild the DNA claims from the Hospital Outpatient Activity report table, test them under three denominators, and cross-check the CSV against the report table.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import common as c
pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.width = 200


In [2]:
op = pd.read_parquet(c.SILVER / 'op_summary1_appointments.parquet').set_index('year')
# Components must sum to the published total in every year.
comp = op[['attendances', 'dnas', 'patient_cancellations', 'hospital_cancellations', 'unknown']].sum(axis=1) - op.total
print('max |components - total|:', comp.abs().max()); assert comp.abs().max() == 0

max |components - total|: 0.0


In [3]:
a, b, o = op.loc['2025-26'], op.loc['2024-25'], op.loc['2019-20']
pc = lambda x, y: 100 * (x / y - 1)
claims = pd.DataFrame([
    ('appointments vs 2024-25 (%)', 2.9, pc(a.total, b.total)),
    ('appointments vs 2019-20 (%)', 20.3, pc(a.total, o.total)),
    ('DNAs vs 2024-25 (%)', 0.1, pc(a.dnas, b.dnas)),
    ('DNAs vs 2019-20 (%)', 5.9, pc(a.dnas, o.dnas)),
    ('DNA rate 2025-26 (%)', 5.4, 100 * a.dnas / a.total),
    ('DNA rate 2019-20 (%)', 6.2, 100 * o.dnas / o.total)], columns=['claim', 'stated', 'rebuilt'])
claims['agrees_at_stated_precision'] = (claims.rebuilt.round(1) == claims.stated)
claims

,claim,stated,rebuilt,agrees_at_stated_precision
0,appointments vs 2024-25 (%),2.9000,2.9215,True
1,appointments vs 2019-20 (%),20.3000,20.3238,True
2,DNAs vs 2024-25 (%),0.1000,0.0856,True
3,DNAs vs 2019-20 (%),5.9000,5.9265,True
4,DNA rate 2025-26 (%),5.4000,5.4226,True
5,DNA rate 2019-20 (%),6.2000,6.1596,True


In [4]:
rates = pd.DataFrame({'dna_rate_all_appointments': 100 * op.dnas / op.total,
                      'dna_rate_attended_or_dna': 100 * op.dnas / (op.attendances + op.dnas),
                      'dna_rate_excluding_unknown': 100 * op.dnas / (op.total - op.unknown)})
rates.round(3)

,dna_rate_all_appointments,dna_rate_attended_or_dna,dna_rate_excluding_unknown
year,,,
2015-16,6.6370,7.7560,6.7240
2016-17,6.6940,7.7910,6.7150
2017-18,6.6880,7.8640,6.7110
2018-19,6.4200,7.5900,6.4490
2019-20,6.1600,7.3910,6.1870
2020-21,5.5360,6.7110,5.5690
2021-22,6.3980,7.5720,6.4250
2022-23,6.4300,7.7000,6.4580
2023-24,5.9210,7.1190,5.9480


In [5]:
# What the DNA count would have been at an earlier rate, on this year's volume.
cf = pd.Series({'at 2019-20 rate': o.dnas / o.total * a.total, 'at 2024-25 rate': b.dnas / b.total * a.total, 'actual': a.dnas})
print(cf.round(0).astype(int).to_string()); print('fewer DNAs than at the 2019-20 rate:', int(cf['at 2019-20 rate'] - a.dnas))
below = [y for y in op.index if rates.dna_rate_all_appointments[y] < rates.dna_rate_all_appointments['2019-20']]
print('years with a lower rate than 2019-20:', below)

at 2019-20 rate    9258962
at 2024-25 rate    8382038
actual             8151086
fewer DNAs than at the 2019-20 rate: 1107876
years with a lower rate than 2019-20: ['2020-21', '2023-24', '2024-25', '2025-26']


In [6]:
# Cross-check: the CSV's all-genders age bands should sum to the report table's attendances.
csv = pd.read_csv(c.OP_AGE_GENDER)
ag = csv[(csv.MAINSPEF_CODE == 'All') & csv.MEASURE.str.endswith('All Genders')]
xc = ag.groupby('MEASURE_TYPE').MEASURE_VALUE.sum()
print(xc.to_string()); print('report table attendances:', int(a.attendances))
assert xc['All Attendances by Age and Gender'] == a.attendances

MEASURE_TYPE
All Attendances by Age and Gender      116891346
First Attendances by Age and Gender     38641885
report table attendances: 116891346


In [7]:
claims.to_parquet(c.GOLD / 'dna_claims_rebuilt.parquet', index=False)
rates.reset_index().to_parquet(c.GOLD / 'dna_rate_by_denominator.parquet', index=False)